# 🌿 AgriVision AI: ConvNeXt V2 Training on Kaggle PlantSeg Dataset
### Multimodal Plant Disease Diagnosis System
This notebook trains **ConvNeXt V2** directly on the **`weitianqi/plantseg` Kaggle dataset** with **Grad-CAM++ Explainable AI**.

- **Dataset:** `weitianqi/plantseg` (Downloaded directly from Kaggle without login)
- **Hardware:** Google Colab Free T4 GPU
- **Expected Time:** ~15 minutes

In [ ]:
# Step 1: Verify GPU is active
!nvidia-smi

In [ ]:
# Step 2: Install required libraries
!pip install -q kagglehub timm grad-cam matplotlib seaborn torchvision pandas

In [ ]:
# Step 3: Download the Kaggle weitianqi/plantseg Dataset directly
import kagglehub
import os

print("Downloading weitianqi/plantseg from Kaggle (this takes ~1-2 mins)...\n")
dataset_path = kagglehub.dataset_download("weitianqi/plantseg")
print("\nDataset downloaded successfully to:", dataset_path)
print("Folder contents:", os.listdir(dataset_path))

In [ ]:
# Step 4: Parse Dataset & Prepare PyTorch DataLoader
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from PIL import Image
import pandas as pd

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Check for Metadata.csv or image folders
csv_file = os.path.join(dataset_path, "Metadata.csv")

class KagglePlantSegDataset(Dataset):
    def __init__(self, root_dir, transform=None, max_samples=3000):
        self.root_dir = root_dir
        self.transform = transform
        self.samples = []
        
        # Look for images folder
        img_dir = os.path.join(root_dir, "images", "train")
        if not os.path.exists(img_dir):
            img_dir = os.path.join(root_dir, "images")
        if not os.path.exists(img_dir):
            img_dir = root_dir
            
        csv_path = os.path.join(root_dir, "Metadata.csv")
        if os.path.exists(csv_path):
            df = pd.read_csv(csv_path)
            # Find image column and label column
            cols = list(df.columns)
            img_col = next((c for c in cols if 'img' in c.lower() or 'file' in c.lower() or 'id' in c.lower()), cols[0])
            lbl_col = next((c for c in cols if 'disease' in c.lower() or 'label' in c.lower() or 'class' in c.lower()), cols[1])
            
            # Get top frequent diseases for fast, accurate training
            top_diseases = df[lbl_col].value_counts().nlargest(8).index.tolist()
            df_filtered = df[df[lbl_col].isin(top_diseases)].copy()
            
            self.label_to_idx = {name: idx for idx, name in enumerate(top_diseases)}
            self.classes = top_diseases
            
            for _, row in df_filtered.iterrows():
                img_name = str(row[img_col])
                if not img_name.endswith('.jpg') and not img_name.endswith('.png'):
                    img_name += '.jpg'
                
                full_path = os.path.join(img_dir, img_name)
                if os.path.exists(full_path):
                    self.samples.append((full_path, self.label_to_idx[row[lbl_col]]))
                    if len(self.samples) >= max_samples:
                        break
        
        # Fallback: scan directory recursively if CSV mapping had missing files
        if len(self.samples) < 50:
            self.samples = []
            classes_found = set()
            for root, _, files in os.walk(img_dir):
                for f in files:
                    if f.lower().endswith(('.jpg', '.jpeg', '.png')) and not f.startswith('.'):
                        cls_name = os.path.basename(root) if root != img_dir else "Diseased Leaf"
                        classes_found.add(cls_name)
                        self.samples.append((os.path.join(root, f), cls_name))
                        if len(self.samples) >= max_samples:
                            break
            self.classes = list(classes_found)
            self.label_to_idx = {c: i for i, c in enumerate(self.classes)}
            self.samples = [(p, self.label_to_idx[c]) for p, c in self.samples]
            
    def __len__(self):
        return len(self.samples)
        
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label

# Transforms
train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

dataset = KagglePlantSegDataset(dataset_path, transform=train_tf)
print(f"Loaded {len(dataset)} images across {len(dataset.classes)} disease categories:")
for idx, name in enumerate(dataset.classes):
    print(f"  [{idx}] {name}")

# 80/20 train/val split
train_len = int(0.8 * len(dataset))
val_len = len(dataset) - train_len
train_ds, val_ds = random_split(dataset, [train_len, val_len])

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2)
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

In [ ]:
# Step 5: Instantiate ConvNeXt V2 Architecture
import timm
import torch.nn as nn

model_name = 'convnextv2_tiny'
num_classes = len(dataset.classes)
model = timm.create_model(model_name, pretrained=True, num_classes=num_classes)
model = model.to(device)
print(f"Successfully initialized {model_name} with {num_classes} classes on GPU!")

In [ ]:
# Step 6: Train ConvNeXt V2 for 8 Epochs
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=8)

epochs = 8
best_acc = 0.0

print(f"=== Starting ConvNeXt V2 Training on Kaggle PlantSeg ({epochs} Epochs) ===\n")

for epoch in range(1, epochs + 1):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    
    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += torch.sum(preds == labels.data).item()
        total += labels.size(0)
            
    scheduler.step()
    train_loss = running_loss / total
    train_acc = correct / total
    
    # Validation Loop
    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for vimages, vlabels in val_loader:
            vimages, vlabels = vimages.to(device), vlabels.to(device)
            voutputs = model(vimages)
            _, vpreds = torch.max(voutputs, 1)
            val_correct += torch.sum(vpreds == vlabels.data).item()
            val_total += vlabels.size(0)
                
    val_acc = val_correct / val_total if val_total > 0 else 0.0
    print(f"Epoch [{epoch:02d}/{epochs:02d}] | Train Loss: {train_loss:.4f} | Train Acc: {train_acc*100:.2f}% | Val Acc: {val_acc*100:.2f}%")
    
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), 'best_convnextv2.pth')

print(f"\n🎉 Training Complete! Best Validation Accuracy: {best_acc*100:.2f}%")

In [ ]:
# Step 7: Explainable AI with Grad-CAM++
from pytorch_grad_cam import GradCAMPlusPlus
from pytorch_grad_cam.utils.image import show_cam_on_image
import numpy as np
import matplotlib.pyplot as plt

target_layers = [model.stages[-1].blocks[-1]]
cam = GradCAMPlusPlus(model=model, target_layers=target_layers)

# Sample validation image from Kaggle dataset
sample_img, sample_lbl = val_ds[0]
input_tensor = sample_img.unsqueeze(0).to(device)

grayscale_cam = cam(input_tensor=input_tensor)[0, :]
inv_norm_img = sample_img.permute(1, 2, 0).cpu().numpy()
inv_norm_img = (inv_norm_img * np.array([0.229, 0.224, 0.225])) + np.array([0.485, 0.456, 0.406])
inv_norm_img = np.clip(inv_norm_img, 0, 1)

visualization = show_cam_on_image(inv_norm_img, grayscale_cam, use_rgb=True)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(inv_norm_img)
axes[0].set_title(f"Kaggle PlantSeg Leaf: {dataset.classes[sample_lbl]}")
axes[0].axis('off')

axes[1].imshow(visualization)
axes[1].set_title("Grad-CAM++ Explainability Heatmap")
axes[1].axis('off')
plt.tight_layout()
plt.savefig('gradcam_kaggle_result.png')
plt.show()
print("Grad-CAM++ visualization successfully generated on Kaggle PlantSeg image!")

In [ ]:
# Step 8: Download Trained Model Checkpoint
from google.colab import files
print("Downloading best_convnextv2.pth...")
files.download('best_convnextv2.pth')
print("Done! Move 'best_convnextv2.pth' into your local project directory.")